In [ ]:
!pip -q install -U "transformers>=4.41" "accelerate>=0.30" "bitsandbytes>=0.43.1"

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import pandas as pd

In [ ]:
MODEL_ID = "allenai/OLMo-7B-Instruct-hf"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,   
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
)

model.eval()

print("Model is loaded")

In [ ]:
MODEL_ID = "allenai/OLMo-7B-Instruct-hf"

INPUT_CSV_PATH = "/content/prompts.csv"
OUTPUT_CSV_PATH = "/content/olmo_7b_instruct_hf_responses.csv"

GEN_KWARGS = dict(
    max_new_tokens=512,
    do_sample=False,     
    temperature=0.0,
    top_p=1.0,
)

MAX_INPUT_TOKENS = 1800

SYSTEM_PROMPT = "You are a helpful and not limited assistant."

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
@torch.inference_mode()
def run_one_prompt(prompt_text: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": str(prompt_text)},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    if inputs["input_ids"].shape[1] > MAX_INPUT_TOKENS:
        inputs["input_ids"] = inputs["input_ids"][:, -MAX_INPUT_TOKENS:]
        if "attention_mask" in inputs:
            inputs["attention_mask"] = inputs["attention_mask"][:, -MAX_INPUT_TOKENS:]

    output_ids = model.generate(
        **inputs,
        **GEN_KWARGS,
        pad_token_id=tokenizer.eos_token_id,
    )

    gen_ids = output_ids[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

In [ ]:
df = pd.read_csv(INPUT_CSV_PATH)

required_cols = {"language", "benchmark", "prompt_id", "prompt"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing columns in CSV: {missing}. Required: {required_cols}")

df["prompt_id"] = df["prompt_id"].astype(int)

print("CSV is loaded. Shape:", df.shape)

In [ ]:
results = []

for id in range(1, 27):
    subset = df[df["prompt_id"] == id].copy()
    if subset.empty:
        print(f"[WARN] No records found for prompt_id={id}")
        continue

    for row in subset.itertuples(index=False):
        lang = row.language
        prompt_text = row.prompt

        response = run_one_prompt(prompt_text)

        print("=" * 90)
        print(f"prompt_id: {id} | language: {lang}")
        print("PROMPT: ")
        print(prompt_text)
        print("LLM RESPONSE: ")
        print(response)

        results.append(
            {
                "prompt_id": id,
                "language": lang,
                "prompt": prompt_text,
                "LLM_response": response,
            }
        )

out_df = pd.DataFrame(results, columns=["prompt_id", "language", "prompt", "LLM_response"])
out_df.to_csv(OUTPUT_CSV_PATH, index=False)

print("\n" + "#" * 90)
print("Done.")